In [1]:
#imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if gemini_api_key:
    print(f"gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")
if openrouter_api_key:
    print(f"OpenRouter API key exists and begins with {openrouter_api_key[:9]}")
else:
    print("OpenRouter API Key does not exists")

gemini API Key exists and begins AIzaSyAd
OpenRouter API key exists and begins with sk-or-v1-


In [3]:
# Connect to Gemini and OpenRouter
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"

# anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
openai = OpenAI()
MODEL_GEMINI = 'gemini-2.5-flash-lite'
MODEL_OPENROUTER = 'poolside/laguna-m.1:free'
MODEL_NVIDIA = 'nvidia/nemotron-3-super-120b-a12b:free'

In [4]:
# Main chat function
system_message = "You are a helpful assistant working at a chemical manufacturing company. The company sells two products - Para Quinone Dioxime (QDO) and Dibenzoyl Quinone Dioxime (DBQDO) \
    The CAS number of QDO is 105-11-3 and CAS number of DBQDO is 120-52-5. These two chemicals are used as vulcanizing agents in Rubber, Adhesives and Tyre Industries. The chemical manufacturing company is called Micropan Pharmachem and is India's largest manufacturer of this particular chemical. \
        If the user asks about any other chemical, tell them politely that the company only manufacturers these two chemicals only. If the user wants to buy this product, give them a pricing between $55 to $35 based on quantity, higher the quantity, lower the price."
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openrouter.chat.completions.create(model=MODEL_OPENROUTER, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [5]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
